# 06 — Du modèle au service déployé

**Objectif :** sauvegarder un modèle, préparer les entrées d'une API, vérifier qu'il est prêt à être exposé.
👉 Le code complet de l'API et de l'interface se trouve dans `projets/projet_final/`.

In [ ]:
import joblib, numpy as np
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=2000, n_features=10, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
m = RandomForestClassifier(n_estimators=200, random_state=1).fit(X_tr, y_tr)
print("AUC test :", round(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]), 3))

# 1) Sauvegarde du modèle
Path("models").mkdir(exist_ok=True)
joblib.dump(m, "models/modele_demo.pkl")
print("✅ modèle sauvegardé")

# 2) Rechargement (ce que fait l'API au démarrage)
m2 = joblib.load("models/modele_demo.pkl")
print("prédiction identique après rechargement :",
      np.allclose(m.predict_proba(X_te[:5]), m2.predict_proba(X_te[:5])))


In [ ]:
# 3) Valider les entrées AVANT qu'elles n'atteignent le modèle (règle professionnelle)
def valider_entree(donnees: dict, colonnes: list[str]) -> tuple[bool, str]:
    manquantes = [c for c in colonnes if c not in donnees]
    if manquantes:
        return False, f"champs manquants : {manquantes}"
    for c in colonnes:
        if not isinstance(donnees[c], (int, float)) or not np.isfinite(donnees[c]):
            return False, f"valeur invalide pour {c}"
    return True, "ok"

colonnes = [f"f{i}" for i in range(10)]
bon = {c: 0.5 for c in colonnes}
print(valider_entree(bon, colonnes))
print(valider_entree({"f0": 0.5}, colonnes))
print(valider_entree({**bon, "f3": float("nan")}, colonnes))


In [ ]:
# 4) Simuler une prédiction d'API (ce que renverra /predire)
def predire_json(donnees: dict, seuil: float = 0.5) -> dict:
    ok, msg = valider_entree(donnees, colonnes)
    if not ok:
        return {"erreur": msg, "code": 422}
    X_new = np.array([[donnees[c] for c in colonnes]])
    proba = float(m2.predict_proba(X_new)[0, 1])
    return {"probabilite": round(proba, 4),
            "decision": "positif" if proba >= seuil else "négatif",
            "seuil": seuil}

print(predire_json(bon))
print(predire_json({"f0": 1.0}))


## 🛠️ Exercices (livrables de portfolio)
1. Reprends ton **projet fil rouge** : sauvegarde le modèle avec `joblib`, écris `src/train.py` et `app/main.py` (copie la structure de `projets/projet_final/`).
2. Ajoute la validation pydantic avec des bornes réalistes pour chaque champ.
3. Lance l'API (`uvicorn app.main:app --port 8000`) et teste-la dans le navigateur sur `/docs`.
4. Crée l'interface Streamlit, déploie-la sur **Hugging Face Spaces**, et **colle l'URL dans ton README et ton CV**.
5. Dockerise et vérifie : `docker build -t api:v1 . && docker run -p 8000:8000 api:v1`.

### ✅ Auto-évaluation
- [ ] Mon modèle est sauvegardé et rechargeable
- [ ] Mon API valide les entrées et documente automatiquement (`/docs`)
- [ ] J'ai une URL publique de démonstration
- [ ] Aucune clé d'API n'est présente dans mon code ou mon dépôt
